# Phase 1 baseline: pure SUE PEAD

Validates the v1 infrastructure end-to-end **before** the LLM augmentation arrives in Phase 2.

**Hypothesis** (Livnat & Mendenhall 2006, Bernard & Thomas 1989):
post-earnings-announcement drift on standardized unexpected earnings (SUE)
delivers a Sharpe in the 0.6–0.8 range, ~4–8% annualized Q5–Q1 spread on US large caps.

**Pipeline:** DuckDB → `casino.signals.pead.compute_sue` → `casino.backtest.vbt_research.run_parameter_sweep`.

This notebook ships with a **synthetic data fallback** so it runs without a Tiingo key. With a real key and ingested 2015-2023 history it should reproduce the documented edge.

In [1]:
from __future__ import annotations

import json
from decimal import Decimal
from pathlib import Path

import numpy as np
import pandas as pd

from casino.backtest import vbt_research
from casino.config import get_config
from casino.data import store
from casino.signals import pead

cfg = get_config()
store.create_schema()
DATA_OK = bool(cfg.tiingo_api_key)
print('Tiingo key configured:', DATA_OK)

2026-05-03 01:39:53.166 | DEBUG    | casino.data.store:create_schema:149 - create_schema completed for data\casino.duckdb


Tiingo key configured: False


## 1. Load earnings + prices

Live path: query DuckDB for 2015-2023 S&P 500 earnings + OHLCV.

Fallback: synthesize a 6-ticker × 8-year quarterly earnings panel + daily prices so the notebook is reproducible offline.

In [2]:
def load_or_synthesize() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return (earnings_df, prices_wide_df). Earnings columns: ticker, report_date, sue."""
    with store.get_duckdb_conn() as conn:
        n = conn.execute('SELECT COUNT(*) FROM earnings').fetchone()[0]
    if n > 1000:
        # live path
        with store.get_duckdb_conn() as conn:
            earnings = conn.execute(
                """
                SELECT ticker, report_date, actual_eps, consensus_eps
                FROM earnings
                WHERE report_date BETWEEN '2015-01-01' AND '2023-12-31'
                  AND actual_eps IS NOT NULL AND consensus_eps IS NOT NULL
                """
            ).df()
            prices = conn.execute(
                """
                SELECT ticker, ts, close FROM ohlcv
                WHERE ts BETWEEN '2015-01-01' AND '2024-06-30'
                """
            ).df()
        prices_wide = prices.pivot(index='ts', columns='ticker', values='close').sort_index()
        return earnings, prices_wide
    # fallback: synthesize
    rng = np.random.default_rng(seed=2026)
    tickers = [f'SYN{i}' for i in range(8)]
    n_days = 252 * 5
    dates = pd.date_range('2019-01-02', periods=n_days, freq='B', tz='UTC')
    drifts = np.linspace(0.0006, -0.0006, len(tickers))
    prices_wide = pd.DataFrame(
        {t: 100 * np.cumprod(1 + rng.normal(mu, 0.012, n_days)) for t, mu in zip(tickers, drifts)},
        index=dates,
    )
    rows = []
    qstarts = pd.date_range('2019-01-31', periods=20, freq='QE', tz='UTC')
    for ticker, mu in zip(tickers, drifts):
        for i, q in enumerate(qstarts):
            base_consensus = 2.00
            actual = base_consensus + (mu * 1500) + rng.normal(0, 0.04)
            rows.append({
                'ticker': ticker,
                'report_date': q,
                'actual_eps': actual,
                'consensus_eps': base_consensus,
            })
    earnings = pd.DataFrame(rows)
    # also persist synthesized earnings for the live SUE path
    persist = [
        {
            'ticker': r.ticker, 'report_date': r.report_date, 'period_end': r.report_date,
            'actual_eps': float(r.actual_eps), 'consensus_eps': float(r.consensus_eps),
            'revenue': 0.0, 'source': 'synthetic',
        }
        for r in earnings.itertuples()
    ]
    store.upsert_earnings(persist)
    return earnings, prices_wide

earnings_df, prices_wide = load_or_synthesize()
print('earnings rows:', len(earnings_df), '| price shape:', prices_wide.shape)
earnings_df.head()

earnings rows: 160 | price shape: (1260, 8)


,ticker,report_date,actual_eps,consensus_eps
0,SYN0,2019-03-31 00:00:00+00:00,2.942354,2.0
1,SYN0,2019-06-30 00:00:00+00:00,2.900421,2.0
2,SYN0,2019-09-30 00:00:00+00:00,2.875919,2.0
3,SYN0,2019-12-31 00:00:00+00:00,2.949896,2.0
4,SYN0,2020-03-31 00:00:00+00:00,2.906547,2.0


## 2. Compute SUE for every (ticker, report_date)

Uses the production `compute_sue` so the research path matches the live path exactly.

In [3]:
def compute_all_sue(earnings: pd.DataFrame) -> pd.DataFrame:
    out = []
    for r in earnings.itertuples():
        sue = pead.compute_sue(
            r.ticker,
            actual_eps=Decimal(str(r.actual_eps)),
            consensus_eps=Decimal(str(r.consensus_eps)),
            as_of_date=r.report_date.to_pydatetime() if hasattr(r.report_date, 'to_pydatetime') else r.report_date,
        )
        out.append({'ticker': r.ticker, 'report_date': r.report_date, 'sue': sue})
    return pd.DataFrame(out).dropna()

sue_df = compute_all_sue(earnings_df)
print('rows with valid SUE:', len(sue_df))
sue_df.head()

2026-05-03 01:39:54.293 | DEBUG    | casino.signals.pead:compute_sue:137 - insufficient history (0 quarters) for SYN0; using industry std


2026-05-03 01:39:54.323 | DEBUG    | casino.signals.pead:compute_sue:137 - insufficient history (1 quarters) for SYN0; using industry std


2026-05-03 01:39:54.870 | DEBUG    | casino.signals.pead:compute_sue:137 - insufficient history (0 quarters) for SYN1; using industry std


2026-05-03 01:39:54.898 | DEBUG    | casino.signals.pead:compute_sue:137 - insufficient history (1 quarters) for SYN1; using industry std


2026-05-03 01:39:55.489 | DEBUG    | casino.signals.pead:compute_sue:137 - insufficient history (0 quarters) for SYN2; using industry std


...

2026-05-03 01:39:57.814 | DEBUG    | casino.signals.pead:compute_sue:137 - insufficient history (1 quarters) for SYN6; using industry std


2026-05-03 01:39:58.383 | DEBUG    | casino.signals.pead:compute_sue:137 - insufficient history (0 quarters) for SYN7; using industry std


2026-05-03 01:39:58.412 | DEBUG    | casino.signals.pead:compute_sue:137 - insufficient history (1 quarters) for SYN7; using industry std


rows with valid SUE: 160


,ticker,report_date,sue
0,SYN0,2019-03-31 00:00:00+00:00,18.847082
1,SYN0,2019-06-30 00:00:00+00:00,18.008422
2,SYN0,2019-09-30 00:00:00+00:00,29.540827
3,SYN0,2019-12-31 00:00:00+00:00,28.273642
4,SYN0,2020-03-31 00:00:00+00:00,25.857691


## 3. Quintile analysis: forward returns by SUE bucket

For each earnings event, compute the next 5/10/20-day cumulative price return; bucket by SUE quintile; check that Q5 > Q1.

In [4]:
def forward_return(prices_wide: pd.DataFrame, ticker: str, after: pd.Timestamp, horizon_days: int) -> float | None:
    if ticker not in prices_wide.columns:
        return None
    series = prices_wide[ticker].dropna()
    series = series[series.index > after]
    if len(series) < horizon_days + 1:
        return None
    return float(series.iloc[horizon_days] / series.iloc[0] - 1)

for h in (5, 10, 20):
    sue_df[f'fwd_{h}d'] = sue_df.apply(
        lambda r: forward_return(prices_wide, r.ticker, r.report_date, h), axis=1,
    )

if not sue_df.empty:
    sue_df['quintile'] = pd.qcut(sue_df['sue'], q=5, labels=False, duplicates='drop')
    summary = sue_df.groupby('quintile')[['fwd_5d', 'fwd_10d', 'fwd_20d']].mean()
    print(summary)

            fwd_5d   fwd_10d   fwd_20d
quintile                              
0        -0.011862 -0.011604 -0.015119
1         0.001117 -0.003644 -0.005038
2        -0.002894  0.010996  0.009306
3        -0.000220  0.001839 -0.005099
4        -0.007491 -0.012554 -0.012836


## 4. vectorbt backtest of the long-Q5 / short-Q1 strategy

Use `run_parameter_sweep` with one config (the baseline) for now. Returns Sharpe / Sortino / max DD / win rate.

In [5]:
def sue_signal_panel(_universe, _start, _end, *, lookback_quarters: int = 8) -> pd.DataFrame:
    """Project SUE scores into a wide date×ticker frame on the price index.
    Each event's SUE persists for the next `holding_window` business days."""
    holding_window = 5
    panel = pd.DataFrame(np.nan, index=prices_wide.index, columns=prices_wide.columns)
    for r in sue_df.itertuples():
        rd = r.report_date
        if rd not in panel.index:
            after = panel.index[panel.index > rd]
            if after.empty:
                continue
            rd = after[0]
        idx = panel.index.get_loc(rd)
        end_idx = min(idx + holding_window, len(panel) - 1)
        panel.iloc[idx:end_idx, panel.columns.get_loc(r.ticker)] = r.sue
    return panel.ffill(limit=holding_window)

results, csv_path = vbt_research.run_parameter_sweep(
    sue_signal_panel,
    param_grid={'lookback_quarters': [4, 8]},
    universe=list(prices_wide.columns),
    prices=prices_wide,
    start_date=prices_wide.index.min().to_pydatetime(),
    end_date=prices_wide.index.max().to_pydatetime(),
    cost_bps=7.5,
    save_results=True,
    output_dir=Path('backtests/results'),
)
results

2026-05-03 01:39:59.502 | INFO     | casino.backtest.vbt_research:run_parameter_sweep:226 - vbt sweep results written to backtests\results\sweep_20260503T063959Z.csv


,lookback_quarters,sharpe,sortino,max_drawdown,win_rate,total_return,universe_count,start_date,end_date,cost_bps
0,4,0.736809,1.247565,-0.082185,0.079365,0.198965,8,2019-01-02T00:00:00+00:00,2023-10-31T00:00:00+00:00,7.5
1,8,0.736809,1.247565,-0.082185,0.079365,0.198965,8,2019-01-02T00:00:00+00:00,2023-10-31T00:00:00+00:00,7.5


## 5. Validate vs academic PEAD literature & export config

**Expected on real S&P 500 data 2015-2023:**
- Sharpe 0.5-0.9
- Q5-Q1 spread 4-8% annualized
- Top-quintile (SUE > ~3) outperforms bottom by 5-10 bps/day for 30-60 days

If the synthetic-data path is in use, results will be noisier and these thresholds will not be met — that is expected and only proves the harness wires up. Re-run with real data.

In [6]:
best = results.iloc[0].to_dict() if not results.empty else {}
Path('backtests').mkdir(exist_ok=True)
config_out = {
    'phase': 'phase_1_baseline_sue',
    'best_params': {k: v for k, v in best.items() if k in {'lookback_quarters'}},
    'metrics': {k: best.get(k) for k in ('sharpe', 'sortino', 'max_drawdown', 'win_rate', 'total_return')},
    'cost_bps': 7.5,
    'data_source': 'duckdb' if DATA_OK else 'synthetic_fallback',
    'literature_targets': {'sharpe_low': 0.5, 'sharpe_high': 0.9, 'q5_minus_q1_pct_annual': '4-8'},
}
Path('backtests/baseline_config.json').write_text(json.dumps(config_out, indent=2, default=str), encoding='utf-8')
print(json.dumps(config_out, indent=2, default=str))

{
  "phase": "phase_1_baseline_sue",
  "best_params": {
    "lookback_quarters": 4
  },
  "metrics": {
    "sharpe": 0.736809488513823,
    "sortino": 1.2475654390035775,
    "max_drawdown": -0.0821849155890998,
    "win_rate": 0.07936507936507936,
    "total_return": 0.19896457786534572
  },
  "cost_bps": 7.5,
  "data_source": "synthetic_fallback",
  "literature_targets": {
    "sharpe_low": 0.5,
    "sharpe_high": 0.9,
    "q5_minus_q1_pct_annual": "4-8"
  }
}
